# **Hito 1 · Baselines de recuperación: TF-IDF + similitud coseno y BM25**

---
**Proyecto PLN (UC3M, 2026/27)** — Chatbot RAG sobre documentación de misiones espaciales.

**Issue #4:** *[02] Implementar TF-IDF + similitud coseno + BM25*

* Generar un **ranking por consulta** con TF-IDF y similitud coseno y otro con BM25.
* Documentar la **implementación**, la **variante de BM25** y los valores de **k1** y **b**.

---

**Entrada** (generada por `01_preprocesado_spacy.ipynb`): `salidas_preprocesado/fragmentos_preprocesados.jsonl`, `diccionario.dict`, `ngramas.phrases` y `config_preprocesado.json`; y `queries.jsonl` en la raíz del repositorio cuando exista.

**Salida** (carpeta `resultados/`, solo si existe `queries.jsonl`):

| Fichero | Contenido |
|---|---|
| `rankings_tfidf.jsonl` | Por consulta: los 10 primeros **documentos** y los 10 primeros **fragmentos** con TF-IDF + coseno |
| `rankings_bm25.jsonl` | Lo mismo con BM25 |
| `config_baselines.json` | Parámetros usados (variante, `k1`, `b`, K) y versiones de las librerías |

**Base:** sección 4 del notebook *Text Vectorization I* (ejercicios 11 a 13). El código reutilizable está en `scripts/baselines.py`, para que el Hito 2 pueda importar los mismos baselines (comparación y búsqueda híbrida). Las adaptaciones a nuestro proyecto están marcadas con 🆕.

## **0. Configuración**

In [1]:
import sys
import json
import time
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 90)

IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = '..'   # raíz del repositorio cuando el notebook se ejecuta en local desde notebooks/
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/My Drive/uc3m-pln-rag-2026'  # UPDATE THIS ACCORDING TO WHERE YOU STORED THE REPOSITORY!!!!
    %pip install --quiet gensim wordninja bm25s

REPO_DIR = Path(REPO_DIR)
FRAGMENTS_PATH = REPO_DIR / 'fragmentos_base' / 'misiones_300.jsonl'
PREPROCESSED_DIR = REPO_DIR / 'salidas_preprocesado'
QUERIES_PATH = REPO_DIR / 'queries.jsonl'
OUT_DIR = REPO_DIR / 'resultados'

# El código de los baselines está en scripts/baselines.py
sys.path.insert(0, str(REPO_DIR / 'scripts'))
from baselines import (QueryPreprocessor, TfidfRetriever, BM25Retriever, check_consistency,
                       rank_documents, rank_chunks, build_rankings, write_jsonl)

Todos los parámetros de los baselines están juntos aquí, para documentarlos en `MILESTONE_1.md`.

In [2]:
CONFIG = {
    'top_k': 10,               # documentos (y fragmentos) que se guardan por consulta: cubre K = 5 y K = 10
    'bm25_method': 'lucene',   # variante de BM25 (la de bm25s por defecto)
    'bm25_k1': 1.5,            # saturación de la frecuencia del término (igual que en Text Vectorization I)
    'bm25_b': 0.75,            # normalización por longitud del fragmento
}

## **1. Carga de los fragmentos preprocesados**

Los tokens son los que guardó el notebook de preprocesado. Del fichero de fragmentos original solo necesitamos el título y la URL (para mostrar los resultados) y el texto (para la comprobación de la sección 2).

In [3]:
frags = pd.read_json(PREPROCESSED_DIR / 'fragmentos_preprocesados.jsonl', lines=True)
meta = pd.read_json(FRAGMENTS_PATH, lines=True)[['chunk_id', 'doc_id', 'title', 'url', 'source', 'text']]

assert frags.chunk_id.is_unique
assert frags.chunk_id.tolist() == meta.chunk_id.tolist(), 'Los fragmentos preprocesados no corresponden a fragmentos_base'

corpus = frags.tokens.tolist()         # lista de listas de tokens: una por fragmento
chunk_ids = frags.chunk_id.tolist()
doc_ids = frags.doc_id.tolist()
doc_info = meta.drop_duplicates('doc_id').set_index('doc_id')[['title', 'url', 'source']]

lengths = np.array([len(tokens) for tokens in corpus])
print(f'{len(corpus):,} fragmentos de {len(doc_info):,} documentos')
print(f'Longitud media de un fragmento: {lengths.mean():.1f} tokens (avgdl de BM25). Fragmentos sin tokens: {(lengths == 0).sum()}')

18,992 fragmentos de 3,314 documentos
Longitud media de un fragmento: 134.8 tokens (avgdl de BM25). Fragmentos sin tokens: 1


## **2. Preprocesado de las consultas**

Las consultas tienen que preprocesarse **exactamente igual** que los fragmentos (sección 4.1 de *Text Vectorization I*). `QueryPreprocessor` aplica los mismos pasos que el notebook `01_preprocesado_spacy` (limpieza, spaCy, filtro por categoría gramatical y stopwords) y carga de `salidas_preprocesado/` la misma configuración, el mismo modelo de n-gramas y el mismo diccionario.

**Comprobación de consistencia** 🆕: como esas funciones están ahora también en `scripts/baselines.py`, comprobamos que al aplicarlas al texto de 200 fragmentos al azar se obtienen exactamente los tokens guardados. Si alguien cambia el preprocesado en un sitio y no en el otro, esta celda falla.

In [4]:
preprocess_queries = QueryPreprocessor(PREPROCESSED_DIR)
D = preprocess_queries.dictionary
print(f'Diccionario: {len(D):,} términos | modelo spaCy: {preprocess_queries.config["spacy_model"]}')

sample = meta.sample(200, random_state=0).index
mismatches = check_consistency(preprocess_queries, meta.text[sample], frags.tokens[sample])
assert not mismatches, f'{len(mismatches)} fragmentos no coinciden con el preprocesado guardado'
print('OK: el preprocesado de consultas reproduce exactamente los tokens del corpus en 200 fragmentos')

Diccionario: 57,168 términos | modelo spaCy: en_core_web_sm
OK: el preprocesado de consultas reproduce exactamente los tokens del corpus en 200 fragmentos


### *Consultas*

Si ya existe `queries.jsonl` (formato del enunciado: `{"qid": ..., "query": ..., "relevant_ids": [...], "source_chunk": ...}`), se usan esas consultas y al final se guardan sus rankings. Si todavía no existe, el notebook usa unas **consultas de ejemplo** solo para enseñar que los baselines funcionan, y no guarda ningún ranking.

In [5]:
if QUERIES_PATH.exists():
    queries_df = pd.read_json(QUERIES_PATH, lines=True, dtype={'qid': str})
    assert {'qid', 'query'} <= set(queries_df.columns) and queries_df.qid.is_unique
    OFFICIAL_QUERIES = True
    print(f'{len(queries_df)} consultas leídas de {QUERIES_PATH.name}')
else:
    queries_df = pd.DataFrame({'query': [
        "Which satellite provided C-band television coverage over North America from a geostationary slot west of Hawaii?",
        "What food and sleep schedule did the crew follow during the first crewed Apollo flight?",
        "How did the team react when they discovered that the asteroid was ejecting particles?",
        "Who launched the Voyager 2 probe and when did it fly past Neptune?",
        "What happened to the mirror of the Hubble Space Telescope after launch?",
        "Spare parts delivered to the space station by shuttle flight STS-129",
    ]})
    queries_df.insert(0, 'qid', [f'demo{n:02d}' for n in range(1, len(queries_df) + 1)])
    OFFICIAL_QUERIES = False
    print(f'No existe {QUERIES_PATH.name} todavía: se usan {len(queries_df)} consultas de ejemplo.')

queries_df['tokens'] = preprocess_queries(queries_df['query'].tolist())
print('Consultas que se quedan sin ningún token:', int((queries_df.tokens.str.len() == 0).sum()))
queries_df[['qid', 'query', 'tokens']].head(10)

No existe queries.jsonl todavía: se usan 6 consultas de ejemplo.
Consultas que se quedan sin ningún token: 0


,qid,query,tokens
0,demo01,Which satellite provided C-band television coverage over North America from a geostati...,"[satellite, provide, band, television, coverage, north_america, geostationary, slot, w..."
1,demo02,What food and sleep schedule did the crew follow during the first crewed Apollo flight?,"[food, sleep, schedule, crew, follow, crew, apollo, flight]"
2,demo03,How did the team react when they discovered that the asteroid was ejecting particles?,"[team, react, discover, asteroid, eject, particle]"
3,demo04,Who launched the Voyager 2 probe and when did it fly past Neptune?,"[launch, voyager, 2, probe, fly, neptune]"
4,demo05,What happened to the mirror of the Hubble Space Telescope after launch?,"[happen, mirror, hubble, space, telescope, launch]"
5,demo06,Spare parts delivered to the space station by shuttle flight STS-129,"[spare, deliver, space, station, shuttle, flight, sts_129]"


## **3. TF-IDF + similitud coseno**

Como en el ejercicio 11 de *Text Vectorization I*, con **Gensim**:

1. `D.doc2bow` convierte cada fragmento en su bolsa de palabras.
2. `TfidfModel` calcula el peso de cada término con sus valores por defecto: $\text{tf} \cdot \log_2(N/\text{df})$, y normaliza cada vector a norma 1.
3. `SparseMatrixSimilarity` guarda la matriz dispersa fragmento-término y calcula la **similitud coseno** de la consulta con todos los fragmentos. Como los vectores tienen norma 1, es un producto escalar.

La consulta se vectoriza con el **mismo diccionario y el mismo modelo TF-IDF** que el corpus.

🆕 $N$ y $\text{df}$ se cuentan sobre **fragmentos**, no sobre documentos, como indica el enunciado del Hito 1.

In [6]:
%%time
tfidf = TfidfRetriever(corpus, D)
print(tfidf.model)
print(f'Matriz TF-IDF: {tfidf.index.index.shape[0]:,} fragmentos x {tfidf.index.index.shape[1]:,} términos, '
      f'{100 * tfidf.index.index.nnz / np.prod(tfidf.index.index.shape):.2f} % de valores no nulos')

TfidfModel<num_docs=18992, num_nnz=1862510>
Matriz TF-IDF: 18,992 fragmentos x 57,168 términos, 0.17 % de valores no nulos
CPU times: total: 5.56 s
Wall time: 5.69 s


## **4. BM25**

Como en la sección 4.3 de *Text Vectorization I*, con la librería [`bm25s`](https://github.com/xhluca/bm25s), que indexa directamente las listas de tokens.

**Variante:** `lucene` (la que usa `bm25s` por defecto, y la de Lucene y Elasticsearch):

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t)\,\frac{f(t,d)}{f(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)} \qquad \text{IDF}(t) = \ln\left(1 + \frac{N - \text{df}(t) + 0.5}{\text{df}(t) + 0.5}\right)$$

Tiene dos diferencias con la fórmula de las diapositivas: no lleva el factor $(k_1 + 1)$ en el numerador, que es una constante y no cambia el orden del ranking, y su IDF nunca es negativo.

**Parámetros:** $k_1 = 1.5$ (saturación de la frecuencia del término) y $b = 0.75$ (normalización por longitud), los mismos valores que en el notebook de la asignatura. $|d|$ es la longitud del **fragmento** en tokens y $\text{avgdl}$ la longitud media de los fragmentos 🆕.

In [7]:
%%time
bm25 = BM25Retriever(corpus, k1=CONFIG['bm25_k1'], b=CONFIG['bm25_b'], method=CONFIG['bm25_method'])
print(f'BM25 ({bm25.method}): k1 = {bm25.k1}, b = {bm25.b}, {bm25.model.scores["num_docs"]:,} fragmentos indexados')

BM25 (lucene): k1 = 1.5, b = 0.75, 18,992 fragmentos indexados
CPU times: total: 1.61 s
Wall time: 1.65 s


**Comprobación de las fórmulas** 🆕: para documentar bien la implementación, recalculamos a mano la puntuación del mejor fragmento de una consulta con las dos fórmulas de arriba y la comparamos con la de las librerías.

In [8]:
from collections import Counter

N, avgdl = len(corpus), lengths.mean()
df = lambda t: D.dfs[D.token2id[t]]            # nº de fragmentos que contienen el término
q_tokens = queries_df.tokens.iloc[0]

# BM25 (variante lucene)
scores = bm25.scores(q_tokens)
best = int(np.argmax(scores))
tf = Counter(corpus[best])
manual = sum(np.log(1 + (N - df(t) + 0.5) / (df(t) + 0.5))
             * tf[t] / (tf[t] + bm25.k1 * (1 - bm25.b + bm25.b * lengths[best] / avgdl)) for t in q_tokens)
print(f'BM25   | {chunk_ids[best]} | bm25s: {scores[best]:.4f} | a mano: {manual:.4f}')
assert np.isclose(scores[best], manual, rtol=1e-4)

# TF-IDF de Gensim + coseno
def tfidf_vector(tokens):
    weights = {t: n * np.log2(N / df(t)) for t, n in Counter(tokens).items()}
    norm = np.sqrt(sum(w ** 2 for w in weights.values()))
    return {t: w / norm for t, w in weights.items()}

scores = tfidf.scores(q_tokens)
best = int(np.argmax(scores))
q_vec, d_vec = tfidf_vector(q_tokens), tfidf_vector(corpus[best])
manual = sum(w * d_vec.get(t, 0) for t, w in q_vec.items())
print(f'TF-IDF | {chunk_ids[best]} | Gensim: {scores[best]:.4f} | a mano: {manual:.4f}')
assert np.isclose(scores[best], manual, rtol=1e-4)

BM25   | doc_3013__0001 | bm25s: 11.4393 | a mano: 11.4393
TF-IDF | doc_0431__0007 | Gensim: 0.2504 | a mano: 0.2504


## **5. Del ranking de fragmentos al ranking de documentos** 🆕

En el notebook de la asignatura cada reseña era un documento. Aquí indexamos fragmentos, pero la evaluación principal del Hito 1 es **a nivel de documento**. `rank_documents` sigue los pasos del enunciado:

1. Se puntúan los fragmentos (todos: son menos de 20.000 y tarda milisegundos).
2. Cada documento recibe la puntuación de su **mejor fragmento**.
3. Se eliminan los documentos repetidos.
4. Se devuelven los **K primeros**.

Puntuar todos los fragmentos, en lugar de pedir solo los K mejores, evita que un documento muy largo (el mayor tiene 1.244 fragmentos) ocupe todos los puestos y deje la lista con menos de K documentos. Un fragmento con puntuación 0 no comparte ningún término con la consulta y no cuenta como recuperado. Los empates se resuelven por el orden del corpus, para que el resultado sea reproducible.

Guardamos también el ranking de fragmentos (`rank_chunks`), que sirve para la evaluación opcional de si el fragmento de origen aparece entre los K primeros.

In [9]:
def show_ranking(qid, retriever, k=5):
    q = queries_df.set_index('qid').loc[qid]
    ranking = pd.DataFrame(rank_documents(retriever.scores(q.tokens), chunk_ids, doc_ids, k),
                           columns=['rank', 'doc_id', 'score', 'best_chunk'])   # vacío si la consulta no tiene tokens
    return ranking.join(doc_info[['title', 'source']], on='doc_id')

qid = queries_df.qid.iloc[min(3, len(queries_df) - 1)]
q = queries_df.set_index('qid').loc[qid]
print('Consulta:', q['query'])
print('Tokens:  ', q.tokens)
print('\nTF-IDF + coseno')
display(show_ranking(qid, tfidf))
print('BM25')
display(show_ranking(qid, bm25))

Consulta: Who launched the Voyager 2 probe and when did it fly past Neptune?
Tokens:   ['launch', 'voyager', '2', 'probe', 'fly', 'neptune']

TF-IDF + coseno


,rank,doc_id,score,best_chunk,title,source
0,1,doc_2366,0.643678,doc_2366__0017,Voyager 2,Wikipedia 1957–1999
1,2,doc_3233,0.385464,doc_3233__0002,Voyager 2 to make closest encounter with Saturn in August,NTRS pilot
2,3,doc_2360,0.359992,doc_2360__0010,Voyager 1,Wikipedia 1957–1999
3,4,doc_3230,0.311420,doc_3230__0002,Voyager 2 Jupiter encounter,NTRS pilot
4,5,doc_3231,0.248181,doc_3231__0093,News conference on Voyager-2-Jupiter encounter,NTRS pilot


BM25


,rank,doc_id,score,best_chunk,title,source
0,1,doc_2366,10.016184,doc_2366__0017,Voyager 2,Wikipedia 1957–1999
1,2,doc_3233,7.318899,doc_3233__0005,Voyager 2 to make closest encounter with Saturn in August,NTRS pilot
2,3,doc_2360,6.479881,doc_2360__0010,Voyager 1,Wikipedia 1957–1999
3,4,doc_2223,5.981309,doc_2223__0017,Pioneer 10,Wikipedia 1957–1999
4,5,doc_3169,5.215983,doc_3169__0029,Juno Launch Press Kit (6.9Mb),JPL PDFs


Primer documento de cada método para todas las consultas cargadas (o las 10 primeras):

In [10]:
rows = []
for q in queries_df.head(10).itertuples():
    top_tfidf = rank_documents(tfidf.scores(q.tokens), chunk_ids, doc_ids, 1)
    top_bm25 = rank_documents(bm25.scores(q.tokens), chunk_ids, doc_ids, 1)
    rows.append({'qid': q.qid, 'query': q.query,
                 'TF-IDF #1': doc_info.title[top_tfidf[0]['doc_id']] if top_tfidf else '(sin resultados)',
                 'BM25 #1': doc_info.title[top_bm25[0]['doc_id']] if top_bm25 else '(sin resultados)'})
pd.DataFrame(rows)

,qid,query,TF-IDF #1,BM25 #1
0,demo01,Which satellite provided C-band television coverage over North America from a geostati...,Kibō (ISS module),GOES 10
1,demo02,What food and sleep schedule did the crew follow during the first crewed Apollo flight?,Apollo 16 Press Kit,Apollo 16 Press Kit
2,demo03,How did the team react when they discovered that the asteroid was ejecting particles?,Deep Space 1 Asteroid Flyby (824KB),OSIRIS-REx Operational Key Decision Points: A Retrospective
3,demo04,Who launched the Voyager 2 probe and when did it fly past Neptune?,Voyager 2,Voyager 2
4,demo05,What happened to the mirror of the Hubble Space Telescope after launch?,Hubble Space Telescope,Hubble Space Telescope
5,demo06,Spare parts delivered to the space station by shuttle flight STS-129,STS-129,STS-129


## **6. Prueba de funcionamiento con los títulos** 🆕

Para comprobar que los dos baselines funcionan sin depender de `queries.jsonl`, usamos como consulta el **título** de 300 documentos al azar y miramos en qué posición sale ese documento.

**Esto no es la evaluación del Hito 1**: el título aparece literalmente en el documento, así que es un caso muy fácil. Solo sirve para detectar errores (por ejemplo, identificadores desalineados) y para ver que los resultados tienen sentido. La evaluación de verdad se hace con las consultas de `queries.jsonl`.

In [11]:
%%time
test_docs = doc_info.sample(300, random_state=0)
title_tokens = preprocess_queries(test_docs.title.tolist())

def position(retriever, tokens, doc_id, k=10):
    ranking = [d['doc_id'] for d in rank_documents(retriever.scores(tokens), chunk_ids, doc_ids, k)]
    return ranking.index(doc_id) + 1 if doc_id in ranking else None

results = []
for name, retriever in [('TF-IDF + coseno', tfidf), ('BM25', bm25)]:
    pos = [position(retriever, tokens, doc_id) for doc_id, tokens in zip(test_docs.index, title_tokens)]
    results.append({'método': name,
                    'en el puesto 1': np.mean([p == 1 for p in pos]),
                    'entre los 5 primeros': np.mean([p is not None and p <= 5 for p in pos]),
                    'entre los 10 primeros': np.mean([p is not None for p in pos]),
                    'MRR@10': np.mean([1 / p if p else 0 for p in pos])})
print('Títulos que se quedan sin ningún token:', sum(len(t) == 0 for t in title_tokens), 'de', len(title_tokens))
pd.DataFrame(results).round(3)

Títulos que se quedan sin ningún token: 7 de 300
CPU times: total: 2.05 s
Wall time: 2.05 s


,método,en el puesto 1,entre los 5 primeros,entre los 10 primeros,MRR@10
0,TF-IDF + coseno,0.60,0.847,0.887,0.697
1,BM25,0.77,0.907,0.927,0.829


## **7. Guardar los rankings**

Un registro por consulta, con este formato:

```json
{"qid": "q01", "query": "...", "tokens": ["..."],
 "docs":   [{"rank": 1, "doc_id": "doc_2366", "score": 10.02, "best_chunk": "doc_2366__0017"}, ...],
 "chunks": [{"rank": 1, "chunk_id": "doc_2366__0017", "doc_id": "doc_2366", "score": 10.02}, ...]}
```

`docs` es el ranking a nivel de documento (el que hay que comparar con `relevant_ids` para Recall@5, Recall@10 y MRR) y `chunks` el de fragmentos (para comparar con `source_chunk`). Solo se guardan si existe `queries.jsonl`.

In [12]:
queries = queries_df[['qid', 'query', 'tokens']].to_dict('records')
rankings = {'tfidf': build_rankings(tfidf, queries, chunk_ids, doc_ids, CONFIG['top_k']),
            'bm25': build_rankings(bm25, queries, chunk_ids, doc_ids, CONFIG['top_k'])}

if OFFICIAL_QUERIES:
    for name, records in rankings.items():
        write_jsonl(OUT_DIR / f'rankings_{name}.jsonl', records)
    with open(OUT_DIR / 'config_baselines.json', 'w', encoding='utf-8', newline='\n') as f:
        json.dump({'config': CONFIG,
                   'tfidf': 'gensim TfidfModel (tf * log2(N/df), norma L2) + SparseMatrixSimilarity (coseno)',
                   'n_queries': len(queries), 'n_fragments': len(corpus), 'n_documents': len(doc_info),
                   'versions': {p: package_version(p) for p in ['gensim', 'bm25s', 'spacy']}},
                  f, ensure_ascii=False, indent=2)
        f.write('\n')
    for path in sorted(OUT_DIR.iterdir()):
        print(f'{path.name:28s} {path.stat().st_size / 1e3:8.1f} kB')
else:
    print('Consultas de ejemplo: no se guarda ningún ranking. Volved a ejecutar el notebook cuando exista queries.jsonl.')

print('\nEjemplo de registro (BM25, 3 primeros documentos):')
print(json.dumps({**rankings['bm25'][0], 'docs': rankings['bm25'][0]['docs'][:3], 'chunks': '...'}, ensure_ascii=False, indent=1))

Consultas de ejemplo: no se guarda ningún ranking. Volved a ejecutar el notebook cuando exista queries.jsonl.

Ejemplo de registro (BM25, 3 primeros documentos):
{
 "qid": "demo01",
 "query": "Which satellite provided C-band television coverage over North America from a geostationary slot west of Hawaii?",
 "tokens": [
  "satellite",
  "provide",
  "band",
  "television",
  "coverage",
  "north_america",
  "geostationary",
  "slot",
  "west",
  "hawaii"
 ],
 "docs": [
  {
   "rank": 1,
   "doc_id": "doc_3013",
   "score": 11.43932,
   "best_chunk": "doc_3013__0001"
  },
  {
   "rank": 2,
   "doc_id": "doc_1301",
   "score": 11.08173,
   "best_chunk": "doc_1301__0001"
  },
  {
   "rank": 3,
   "doc_id": "doc_3107",
   "score": 10.729753,
   "best_chunk": "doc_3107__0001"
  }
 ],
 "chunks": "..."
}


### *Cómo usarlo en el issue de la evaluación (Recall@K y MRR)*

```python
import pandas as pd

queries = pd.read_json('queries.jsonl', lines=True, dtype={'qid': str}).set_index('qid')
rankings = pd.read_json('resultados/rankings_bm25.jsonl', lines=True, dtype={'qid': str}).set_index('qid')

ranked_docs = rankings.docs.map(lambda docs: [d['doc_id'] for d in docs])   # lista ordenada de doc_id por consulta
relevant = queries.relevant_ids.map(set)                                     # conjunto de doc_id relevantes
```

Con `ranked_docs` y `relevant` se pueden aplicar directamente las funciones `recall_at_k` y `mrr` del ejercicio 12 de *Text Vectorization I*.

---
## **Resumen de decisiones (para `MILESTONE_1.md`)**

| Aspecto | Decisión | Origen |
|---|---|---|
| Unidad de indexación | Fragmento: IDF, longitud y `avgdl` se calculan sobre los 18.992 fragmentos | Enunciado del Hito 1 |
| TF-IDF | Gensim `TfidfModel` por defecto: $\text{tf} \cdot \log_2(N/\text{df})$ y norma L2 | TV-I §2.4 |
| Similitud | Coseno con Gensim `SparseMatrixSimilarity` | TV-I §4.2 |
| BM25 | Librería `bm25s`, variante `lucene` | TV-I §4.3 |
| Parámetros de BM25 | $k_1 = 1.5$, $b = 0.75$ | TV-I §4.3 |
| Consultas | Mismo preprocesado, n-gramas y diccionario que los fragmentos, comprobado automáticamente | TV-I §4.1 |
| Ranking de documentos | Puntuación del mejor fragmento, sin repetidos, K primeros; puntuación 0 no cuenta | Enunciado del Hito 1 🆕 |
| Rankings guardados | 10 documentos y 10 fragmentos por consulta y método en `resultados/` | 🆕 |

**Pendiente** (otros issues): ejecutar este notebook con `queries.jsonl`, calcular Recall@5, Recall@10 y MRR, analizar las consultas que fallan los dos métodos y, si se quiere, probar otros valores de $k_1$ y $b$ (ejercicio 13 de *Text Vectorization I*).